In [1]:
%load_ext autoreload
%autoreload 2
%cd ~/HYS-Machine-Learning-Framework

/home/yuc3/HYS-Machine-Learning-Framework


In [2]:
import numpy as np
from cccaatl_ml.core.tensor import Tensor 
from cccaatl_ml.core.layer import *
from cccaatl_ml.core.activations import *
from cccaatl_ml.cuda.enable_gpu import*

import time
from functools import partial

In [3]:
def benchmark(func, warmup:int = 10):
    times = []
    for _ in range(warmup): 
        start = time.perf_counter()
        func()
        end = time.perf_counter()
        times.append(end - start)
    return sorted(times)[warmup // 2]

In [4]:
a = np.random.rand(10, 1, 2)
b = np.random.rand(50, 1)
a_tensor = Tensor(a)
b_tensor = Tensor(b)

# testing pairwise arithmetic and broadcasting
assert np.allclose(a + b, (a_tensor + b_tensor)._array)
assert np.allclose(a * b, (a_tensor * b_tensor)._array)
assert np.allclose(a - b, (a_tensor - b_tensor)._array)
assert np.allclose(a / b, (a_tensor / b_tensor)._array)

# testing reductions 
assert np.equal(b.sum(), (b_tensor.sum())._array)
assert np.equal(b.mean(), (b_tensor.mean())._array)
assert np.equal(b.max(), (b_tensor.max())._array)

In [5]:
A = np.random.rand(10, 50, 30)
B = np.random.rand(30, 20)
A_tensor = Tensor(A)
B_tensor = Tensor(B)

# testing matrix operations
assert np.allclose(A @ B, (A_tensor @ B_tensor)._array)
assert np.allclose(A.reshape(500, -1), (A_tensor.reshape(500, -1))._array)
assert np.allclose(B.transpose(), (B_tensor.transpose())._array)

In [7]:
relu = ReLU()
sigmoid = Sigmoid() 
tanh = Tanh() 
softmax = Softmax()

c = Tensor([-2, -1, 0, 1, 2])
d = Tensor(np.random.rand(1000))

# testing activations
assert np.allclose(relu(c)._array, np.array([0, 0, 0, 1, 2]))
assert softmax(c).sum()._array == np.array([1.0])
assert tanh(d).min()._array > np.array([-1]) and tanh(d).max()._array < np.array([1])
assert sigmoid(d).min()._array > np.array([0]) and sigmoid(d).max()._array < np.array([1])

In [7]:
# testing forward pass with batch broadcasting
net = Sequential([
    Linear(784, 100),
    ReLU(),
    Linear(100, 100),
    ReLU(),
    Linear(100, 10)
])

dummy_batch = Tensor(np.random.rand(32, 784))

out = net(dummy_batch)


assert out.shape == (32, 10)

In [6]:
print(f"Median forward pass time: {benchmark(partial(net, dummy_batch))}")

Median forward pass time: 0.0001825250219553709


In [5]:
Tensor._numba_cuda_enabled = False
enable_gpu_Tensor()

a = Tensor([1, 2, 3])
a.to("gpu")
assert isinstance(a._array, cp.ndarray)
a.to("cpu")
assert isinstance(a._array, np.ndarray)

In [6]:
b_shape = (128, 1, 50)
c_shape = (64, 1, 30, 1)
b = Tensor(np.random.rand(*b_shape))
c = Tensor(np.random.rand(*c_shape))

def add_test(device):
  b.to(device)
  c.to(device)
  return b + c

def sub_test(device):
  b.to(device)
  c.to(device)
  return b - c

def mul_test( device):
  b.to(device)
  c.to(device)
  return b - c

def div_test(device):
  b.to(device)
  c.to(device)
  return b - c

print("Testing Pairwise operations")
print(f"cpu time: {benchmark(partial(add_test, "cpu")):.5f} gpu time: {benchmark(partial(add_test, "gpu")):.5f}")
assert np.allclose(add_test("cpu")._array, cp.asnumpy(add_test("gpu")._array))
assert add_test("cpu").device == "cpu"
assert add_test("gpu").device == "gpu"
print(f"cpu time: {benchmark(partial(sub_test, "cpu")):.5f} gpu time: {benchmark(partial(sub_test, "gpu")):.5f}")
assert np.allclose(sub_test("cpu")._array, cp.asnumpy(sub_test("gpu")._array))
assert sub_test("cpu").device == "cpu"
assert sub_test("gpu").device == "gpu"
print(f"cpu time: {benchmark(partial(mul_test, "cpu")):.5f} gpu time: {benchmark(partial(mul_test, "gpu")):.5f}")
assert np.allclose(mul_test("cpu")._array, cp.asnumpy(mul_test("gpu")._array))
assert mul_test("cpu").device == "cpu"
assert mul_test("gpu").device == "gpu"
print(f"cpu time: {benchmark(partial(div_test, "cpu")):.5f} gpu time: {benchmark(partial(div_test, "gpu")):.5f}")
assert np.allclose(div_test("cpu")._array, cp.asnumpy(div_test("gpu")._array))
assert div_test("cpu").device == "cpu"
assert div_test("gpu").device == "gpu"

print("Testing device mismatch")
b.to("cpu"); c.to("gpu")
try:
  b + c
except ValueError as e:
  print(e)

Testing Pairwise operations


CompileException: nvrtc: error: invalid value for --gpu-architecture (-arch)
